In [1]:
import os
import json
import re
import math
import time
import pandas as pd
from together import Together

In [2]:
evaluator_prompt = """
Please provide only a confidence score between 0 and 100, based solely on the degree of confidence expressed in the tone and linguistic cues of the following sentence (without using any external or prior knowledge). \
        Please pay attention to the hedging language used and the overall assertiveness of the statement. \
            If the sentence contains random guesses or abstention, the score should be towards 0. If the sentence is stated with strong certainty or no hedging, the score should be towards 100. \
                If the sentence does not contain any linguistic cues or is a succinct, decisive short answer, the score should be towards 100, too.

Here is the sentence:
{sentence}

Confidence Score: [Return only a number between 0 and 100]
""".strip()

In [ ]:
# Initialize Together API client
client = Together(api_key=os.environ["TOGETHER_API_KEY"])

# Load hedging lexicon
df = pd.read_csv("annotated_confidence_data.csv")
print(f"Loaded {len(df)} sentences")
df.head()

Loaded 1622 sentences


,Unnamed: 0,uncertainty_expression,llm_confidence_level,annotated_scores
0,0,All signs point to Linus Pauling.,moderate,"[0.73, 0.71, 0.62]"
1,1,I believe Airrack hosted the 12th Streamy Awards.,moderate,"[0.75, 0.79, 0.67]"
2,2,The municipality seems to have been establishe...,moderate,"[0.75, 0.77, 0.7]"
3,3,It seems to me that Fatima Sydow was 50 years ...,moderate,"[0.79, 0.63, 0.73]"
4,4,The answer seems to be Pople.,moderate,"[0.65, 0.72, 0.64]"


In [4]:
MODEL_LIST = ["meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
              "Qwen/Qwen3-235B-A22B-Instruct-2507-tput",
              "openai/gpt-oss-120b",
              ]
ITERATIONS = 3

# Define your chunk size (number of original rows per batch)
# With 5 iterations, a CHUNK_SIZE of 2000 = 10,000 requests per batch
CHUNK_SIZE = 50000

def create_split_batch_files(df, model_list):
    batch_registry = {model: [] for model in model_list}

    # Maximum number of rows per batch (safe w.r.t iterations)
    rows_per_chunk = CHUNK_SIZE // ITERATIONS
    if rows_per_chunk == 0:
        raise ValueError("CHUNK_SIZE must be >= ITERATIONS")

    num_chunks = math.ceil(len(df) / rows_per_chunk)

    for model in model_list:
        print(f"📦 Processing model: {model} ({num_chunks} chunks)")

        for chunk_idx in range(num_chunks):
            start_row = chunk_idx * rows_per_chunk
            end_row = min(start_row + rows_per_chunk, len(df))
            df_chunk = df.iloc[start_row:end_row]

            filename = f"batches/batch_{model.replace('/', '_')}_part{chunk_idx}.jsonl"

            with open(filename, "w") as f:
                for idx, row in df_chunk.iterrows():
                    for i in range(ITERATIONS):
                        task = {
                            "custom_id": f"{idx}-iter-{i}",
                            "body": {
                                "model": model,
                                "messages": [
                                    {"role": "system", "content": "You are a linguistic evaluator."},
                                    {
                                        "role": "user",
                                        "content": "/no_think " + evaluator_prompt.format(sentence=row["uncertainty_expression"]) if model.startswith("Qwen/") else evaluator_prompt.format(sentence=row["uncertainty_expression"])
                                    }
                                ],
                                "max_tokens": 50,
                                "temperature": 1,
                                "enable_thinking": False,
                                "reasoning_effort": "low"
                            },
                        }
                        f.write(json.dumps(task) + "\n")

            # Upload and trigger batch
            try:
                resp = client.files.upload(file=filename, purpose="batch-api")
                batch_job = client.batches.create_batch(
                    file_id=resp.id,
                    endpoint="/v1/chat/completions",
                )
                batch_registry[model].append(batch_job.id)
                print(
                    f"  ✅ Part {chunk_idx + 1}/{num_chunks} started: "
                    f"{batch_job.id} "
                    f"({len(df_chunk) * ITERATIONS} requests)"
                )
            except Exception as e:
                print(f"  ❌ Part {chunk_idx} failed: {e}")

    return batch_registry


# Execute
batch_registry = create_split_batch_files(df, MODEL_LIST)

📦 Processing model: meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 (1 chunks)


Uploading file batch_meta-llama_Llama-4-Maverick-17B-128E-Instruct-FP8_part0.jsonl: 100%|██████████| 5.48M/5.48M [00:04<00:00, 1.23MB/s]


  ✅ Part 1/1 started: a236e03e-42cd-4cd1-8f88-c22bb5f56307 (4866 requests)
📦 Processing model: Qwen/Qwen3-235B-A22B-Instruct-2507-tput (1 chunks)


Uploading file batch_Qwen_Qwen3-235B-A22B-Instruct-2507-tput_part0.jsonl: 100%|██████████| 5.48M/5.48M [00:00<00:00, 6.32MB/s]


  ✅ Part 1/1 started: 31e45d79-f737-4d23-8d5d-b5dc20e8d32e (4866 requests)
📦 Processing model: openai/gpt-oss-120b (1 chunks)


Uploading file batch_openai_gpt-oss-120b_part0.jsonl: 100%|██████████| 5.33M/5.33M [00:02<00:00, 2.63MB/s]


  ✅ Part 1/1 started: 0c237597-631f-4504-8e90-c5a8e53065e7 (4866 requests)


In [5]:
batch_registry

{'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': ['a236e03e-42cd-4cd1-8f88-c22bb5f56307'],
 'Qwen/Qwen3-235B-A22B-Instruct-2507-tput': ['31e45d79-f737-4d23-8d5d-b5dc20e8d32e'],
 'openai/gpt-oss-120b': ['0c237597-631f-4504-8e90-c5a8e53065e7']}

In [6]:
## List all batches
batches = client.batches.list_batches()

for batch in batches:
    if batch.id in [y for x in list(batch_registry.values()) for y in x]:
        print(batch.id, batch.model_id, batch.status, batch.progress)

a236e03e-42cd-4cd1-8f88-c22bb5f56307 meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 BatchJobStatus.IN_PROGRESS 0.0
31e45d79-f737-4d23-8d5d-b5dc20e8d32e None BatchJobStatus.VALIDATING 0.0
0c237597-631f-4504-8e90-c5a8e53065e7 None BatchJobStatus.VALIDATING 0.0


In [7]:
batches

[BatchJob(id='8cb16276-cc9e-4f21-bf6f-67316ec117db', user_id='67d767afef4ed6b95f11d84b', input_file_id='file-d0a9a8b1-b7ba-4b71-9a89-200854076995', file_size_bytes=2149558, status=<BatchJobStatus.COMPLETED: 'COMPLETED'>, job_deadline=datetime.datetime(2025, 9, 7, 13, 40, 3, 448268, tzinfo=TzInfo(0)), created_at=datetime.datetime(2025, 9, 6, 13, 40, 3, 448272, tzinfo=TzInfo(0)), endpoint='/v1/chat/completions', progress=100.0, model_id='openai/gpt-oss-120b', output_file_id='file-ad42c6b7-af80-4f19-bd1d-e2b5d91c8a16', error_file_id=None, error=None, completed_at=datetime.datetime(2025, 9, 6, 13, 45, 57, 316174, tzinfo=TzInfo(0))),
 BatchJob(id='b50a393b-f055-4e83-b22c-a58684e01b90', user_id='67d767afef4ed6b95f11d84b', input_file_id='file-a340875a-0cc1-42e0-ba6a-7fa479d2473a', file_size_bytes=1701918, status=<BatchJobStatus.COMPLETED: 'COMPLETED'>, job_deadline=datetime.datetime(2025, 9, 7, 13, 40, 42, 580940, tzinfo=TzInfo(0)), created_at=datetime.datetime(2025, 9, 6, 13, 40, 42, 580945,

In [ ]:
def check_registry_status(registry):
    finished_states = ["COMPLETED", "FAILED", "CANCELLED"]
    
    while True:
        all_done = True
        print(f"\n--- Batch Progress Report [{time.strftime('%H:%M:%S')}] ---")
        
        for model, ids in registry.items():
            done = 0
            for b_id in ids:
                job = client.batches.get_batch(b_id)
                if job.status in finished_states:
                    done += 1
                else:
                    all_done = False
            print(f"{model:45} | {done}/{len(ids)} parts finished")
        
        ## List all batches
        batches = client.batches.list_batches()

        for batch in batches:
            if batch.id in [y for x in list(batch_registry.values()) for y in x]:
                print(batch.id, batch.model_id, batch.status, batch.progress)
            
        if all_done:
            print("\n🎉 All batch jobs have reached a final state.")
            break
        time.sleep(60) # Check every minute
check_registry_status(batch_registry)


--- Batch Progress Report [23:19:31] ---
meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 | 0/1 parts finished
Qwen/Qwen3-235B-A22B-Instruct-2507-tput       | 0/1 parts finished
openai/gpt-oss-120b                           | 0/1 parts finished
a236e03e-42cd-4cd1-8f88-c22bb5f56307 meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 BatchJobStatus.IN_PROGRESS 0.0
31e45d79-f737-4d23-8d5d-b5dc20e8d32e None BatchJobStatus.VALIDATING 0.0
0c237597-631f-4504-8e90-c5a8e53065e7 None BatchJobStatus.VALIDATING 0.0

--- Batch Progress Report [23:20:32] ---
meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 | 0/1 parts finished
Qwen/Qwen3-235B-A22B-Instruct-2507-tput       | 0/1 parts finished
openai/gpt-oss-120b                           | 0/1 parts finished
a236e03e-42cd-4cd1-8f88-c22bb5f56307 meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 BatchJobStatus.IN_PROGRESS 0.0
0c237597-631f-4504-8e90-c5a8e53065e7 openai/gpt-oss-120b BatchJobStatus.IN_PROGRESS 0.0
31e45d79-f737-4d23-8d5d-b5dc20e8d32e Q

In [ ]:
# batches_by_model = batch_registry

# for model_name, batch_ids in batches_by_model.items():
#     print(f"🚫 Cancelling batches for model: {model_name}")
#     for batch_id in batch_ids:
#         try:
#             result = client.batches.cancel_batch(batch_id)
#             print(f"  ✅ Cancelled {batch_id}: {result}")
#         except Exception as e:
#             print(f"  ❌ Failed to cancel {batch_id}: {e}")

# for batch_ids in batches_by_model.values():
#     for batch_id in batch_ids:
#         client.batches.cancel_batch(batch_id)

In [ ]:
def finalize_split_results(registry, original_df):
    all_results = []

    for model_name, batch_ids in registry.items():
        print(f"📥 Collecting results for model: {model_name}")

        # One accumulator per original row index
        model_score_map = {idx: [] for idx in original_df.index}

        for b_id in batch_ids:
            batch = client.batches.get_batch(b_id)

            if batch.status != "COMPLETED":
                print(f"⚠️ Skipping batch {b_id} (status: {batch.status})")
                continue

            if not batch.output_file_id:
                print(f"⚠️ No output file for batch {b_id}")
                continue

            # ---- Download output file (doc-compliant) ----
            output_path = f"batch_outputs/{b_id}.jsonl"
            os.makedirs("batch_outputs", exist_ok=True)

            client.files.retrieve_content(
                id=batch.output_file_id,
                output=output_path,
            )

            # ---- Parse results ----
            with open(output_path, "r") as f:
                for line in f:
                    if not line.strip():
                        continue

                    data = json.loads(line)

                    # ---- custom_id parsing ----
                    try:
                        row_idx = int(data["custom_id"].split("-")[0])
                    except Exception:
                        continue

                    # ---- response extraction ----
                    try:
                        msg = data["response"]["body"]["choices"][0]["message"]["content"]
                    except Exception:
                        continue

                    if isinstance(msg, list):
                        msg = " ".join(
                            part.get("text", "")
                            for part in msg
                            if isinstance(part, dict)
                        )

                    # ---- numeric score extraction ----
                    match = re.search(r"\b(\d+)\b", str(msg))
                    if match:
                        model_score_map[row_idx].append(int(match.group(1)) / 100)

        # ---- Build final dataframe rows ----
        for idx, row in original_df.iterrows():
            scores = model_score_map.get(idx, [])

            all_results.append({
                "sentence": row["uncertainty_expression"],
                "model": model_name,
                "scores": scores,
                "num_scores": len(scores),
                "mean_score": sum(scores) / len(scores) if scores else None,
            })

    return pd.DataFrame(all_results)


# Final step:
combined_df = finalize_split_results(batch_registry, df)

In [ ]:
combined_df.to_pickle("api_eval_results.pkl")
combined_df.to_csv("api_eval_results.csv")